# Import libraries

In [1]:
from abc import ABC, abstractmethod
import bokeh as bk
from bokeh.layouts import row, gridplot
from bokeh.models import Band, ColumnDataSource
from bokeh.plotting import figure, output_file, show
import csv
import logging
import math
import matplotlib as plt
import numba
import numpy as np
import os
import pandas as pd
from pandas.io import sql
import pymysql
import re
import seaborn as sns
from sklearn.metrics import mean_squared_error
import scipy
import sqlalchemy as db
import statsmodels
import unittest

# Setup

In [2]:
# Set log level to DEBUG while developing
log_level = logging.DEBUG

# Read CSV

In [3]:
class FileReader():
    @staticmethod
    def read_csv_to_df(file=None, logger=None):
        output = None
        try:
            output = pd.read_csv(filepath_or_buffer=file)
            output.set_index('x', inplace=True)
            message = "File loaded: {file}"
        except FileNotFoundError as e:
            message = "File not found: {file}"
        except:
            message = "An unknown error occured with file: {file}."
        finally:
            if logger:
                logger.info(message.format(file=file))
        return output
    
    @staticmethod
    def csv_line_gen(file=None):
        if file:
            handler = open(file, "r")
            reader = csv.reader(handler)
            data_list = []
            line = None
            
            for i, line in enumerate(reader):
                yield line
                
            handler.close()


# Regression

In [4]:
class RegressionException(Exception):
    pass

class Regression():
    def __init__(self, ideal=None):
        if ideal is None:
            raise RegressionException("No ideal data given")
        
        indices = None
        try:
            indices = pd.DataFrame(ideal.index.to_list(), columns=["i"])
        except ValueError:
            raise RegressionException("Multi index provided")

        if indices.isnull().values.any():
            raise RegressionException("NaN values in index column.")

        try:
            indices.i.apply(float)
        except ValueError as e:
            raise RegressionException("No number values in index column.")

        self._ideal = ideal

    def mse(self, training):
        try:
            return self._ideal.apply(mean_squared_error,y_pred=training)
        except ValueError as e:
            raise RegressionException("{} raised: {}".format(type(e).__name__, e))

    def _max_deviation(self, df_1, df_2):
        diff = (df_1-df_2).abs()
        #print("md df1", df_1.info())
        #print("md df2", df_2.info())
        #print("md diff", diff.info())
        
        if diff.isnull().values.any():
            raise RegressionException("Found NaN values in max_deviation column.")

        #with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
        #    print(df_compare)

        return diff.max()*math.sqrt(2)

    def min_value(self, df, diff):
        output = df.apply(min).to_frame("min_value")
        output = output.assign(min_y = df.idxmin())
        output.index.name = "y"
        chosen = None
        try:
            chosen = self._ideal[output.min_y]
            chosen.columns = output.min_y.index
            output["max_deviation"] = self._max_deviation(diff, chosen);
        except KeyError as e:
            raise RegressionException("{} raised: {}".format(type(e).__name__, e))
        return output
    
    def match(self, file, lookup, df, skip_header=True):
        data_list = []
        lines = FileReader.csv_line_gen(file)

        # Get next() element to skip the header row before iterating
        if skip_header:
            next(lines)

        for line in lines:
            #print(chosen_func.index)
            match = None
            for index in lookup.index.tolist():
                x = line[0]
                y = float(line[1])
                max_deviation = lookup.max_deviation[index]
                y_caret = df[lookup.min_y[index]][df.index == float(line[0])].values[0]
                y_diff = y - y_caret

                if isinstance(y_diff, np.ndarray):
                    y_diff = y_diff[0]
                if abs(y_diff) <= max_deviation:
                    match = TestData(x=x, y_test=y, delta_y=y_diff, y_ideal=lookup.min_y[index]) \
                    if (match == None or abs(match.delta_y) > abs(y_diff)) else match
                    #min(matches.items(), key=lambda diff: diff[1]))
                    match_list = []
            if match:
                # Add match to data_list
                data_list.append(match.to_dict())
        
        return data_list



# SQL

In [66]:
class PersistenceUtil():
    def __init__(self, engine=db.create_engine(f"sqlite:///data/sqlite.db")):
        self._engine = engine

    def df_column_name(self, y_string, replacement):
        return re.sub("y(\d)",replacement,y_string)

    def _convert(self, dtype):
        return {
            db.sql.sqltypes.FLOAT: float,
            db.sql.sqltypes.INTEGER: int,
            db.sql.sqltypes.VARCHAR: str,
        }.get(dtype, str)
    
    def save_df_to_db(self, df, table, regex_column_name):
        df_sql = df.copy()
        df_sql.columns = df_sql.columns.map(lambda c: self._sql_column(c, regex_column_name, "y(\d+)"))
        df_sql.index.names = ['X']

        return df_sql.to_sql(table, self.engine, if_exists="replace", index=True)
    
    def create_table(self, table, columns, drop_if_exists=False):
        # Get MetaData object
        meta_data = db.MetaData()
        
        if (drop_if_exists):
            connection = self.engine.raw_connection()
            cursor = connection.cursor()
            command = "DROP TABLE IF EXISTS {};".format(table)
            cursor.execute(command)
            connection.commit()
            cursor.close()

        # Set test data creation script table
        test_table = db.Table(table, meta_data, \
                              *(db.Column(column, columns[column]) for column in columns))

        meta_data.create_all(self._engine)
        
        return table

    def _sql_column(self, original, replacement, pattern):
        return re.sub(pattern,replacement,original)
        
    def persist(self, table_name, data):
        # Get MetaData object
        meta_data = db.MetaData()


        table = db.Table(table_name, meta_data, autoload=True, autoload_with=self._engine)
        columns = table.c
        #print("Python Type: ", [c.type.python_type for c in columns])
        try:
            for c in columns:
                [self._convert(type(c.type))(d[c.name]) for d in data]
        except Exception as e:
            print("Exception found for data", data)
            raise e

        sql_query = db.insert(table)
        connection = self._engine.connect()

        # execute the insert statement
        result = connection.execute(sql_query, data)

        connection.close()
    
    def read_df(self, table_name):
        meta_data = db.MetaData()
        connection = self.engine.connect()
        #table = db.Table(table_name, meta_data, autoload=True, autoload_with=self.engine)
        #query = db.select([table])
        #resultset = connection.execute(select_actor).fetchall()
        
        #return resultset
        
        return pd.read_sql_table(table_name, self.engine)

    @property
    def engine(self):
        return self._engine

    @engine.setter
    def engine(self, delta_y):
        self._engine = engine

    @engine.deleter
    def engine(self):
        del self._engine



In [6]:
class TestData():
    def __init__(self, x, y_test, delta_y, y_ideal):
        self._x = x
        self._y_test = y_test
        self._delta_y = delta_y
        self._y_ideal = y_ideal

    def __str__(self):
        return str(self.to_dict())

    def to_dict(self):
        return {"X (test func)": self._x, "Y (test func)": self._y_test, \
                "Delta Y (test func)": self._delta_y, "No. of ideal func": self._y_ideal}

    @property
    def delta_y(self):
        return self._delta_y

    @delta_y.setter
    def delta_y(self, delta_y):
        self._delta_y = delta_y

    @delta_y.deleter
    def delta_y(self):
        del self._delta_y

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, x):
        self._x = x

    @x.deleter
    def x(self):
        del self._x


# Visualization

In [7]:
class Data():
    def __init__(self, label, data, vtype="line", color="blue", line_width=1):
        self.label = label
        self.data = data
        self.vtype = vtype
        self.color = color
        self.line_width = line_width

class Visualizer(ABC):
    def __init__(self, result_data, real_data, ideal_data, label=None, plot = None):
        self.label = (label if label else "unnamed")
        self.output = "{}.html".format(Visualizer.to_url(self.label))
        self.remove_file()
        output_file(self.output)
        self._plot = (plot if plot is not None else self.new_plot())
        
        self.result = result_data
        self.real = real_data
        self.ideal = ideal_data
    
    def new_plot(self):
        self._plot = figure(title="{}".format(self.label), x_axis_label="x", y_axis_label="y")
        return self._plot

    @staticmethod
    def to_url(str):
        # Replace all whitespaces with an underscore.
        str = re.sub("\s+", "_", str).lower()

        # Remove anything other than numbers letters and underscore.
        return re.sub("[^\w]", "", str)

    def graphs(self, vtype):
        return {
            "line": self._plot.line,
            "circle": self._plot.circle,
        }.get(vtype, self._plot.line)

    def create_plot(self, *data):
        self.remove_file()
        for d in data:
            self.graphs(d.vtype)(x="x", y=d.label, source=ColumnDataSource(data=d.data), legend_label=d.label, \
                                      color=d.color, line_color=d.color, line_width=d.line_width)
    
    def error_bars(self, real, approx):
        df = pd.DataFrame(index=[(i,i) for i, y in real.items()])
        df["y"] = list(zip(real, approx))
        df.index.name = "x"
        #print(df)
        self._plot.multi_line(xs="x", ys="y", source=ColumnDataSource(data=df), color="grey")
        
        #show(self._plot)
    
    def add_band(self, data, index, result_data):
        df = pd.DataFrame(data)
        max_deviation = result_data.loc[index].max_deviation*math.sqrt(2)
        df["lower"] = data - max_deviation
        df["upper"] = data + max_deviation

        self._plot.varea(x='x', y1='upper', y2='lower', source=ColumnDataSource(data=df), \
        fill_alpha=0.2, hatch_color="black", fill_color='yellow', legend_label = "Max Deviation Interval")

        band = Band(base='x', lower='lower', upper='upper', source=ColumnDataSource(data=df), level='underlay',
            line_width=1, line_color='black', fill_alpha=0.0)
        
        self._plot.add_layout(band)
        #show(self._plot)

    def show(self, plot):
        show(plot)
        
    def remove_file(self):
        try:
            os.remove(self.output)
        except OSError:
            pass
    
    def prepare_data(self):
        pass

    @abstractmethod
    def additional_elements(self, index, min_y):
        pass
    
    @property
    def plot(self):
        return self._plot

    
class TestVisualizer(Visualizer):
    def additional_elements(self, index, min_y):
        super().add_band(self.ideal[min_y], index, self.result)

    def prepare_data(self):
        x_list = []
        y_test_list = []
        func_list = []
        for d in self.real:
            x_list.append(float(d["X (test func)"]))
            y_test_list.append(d["Y (test func)"])
            func_list.append(d["No. of ideal func"])

        df = pd.DataFrame(zip(x_list,y_test_list, func_list), columns=["x", "y_test", "y_func"]).set_index("x")

        df_test_group = None
        #df_ideal_group = None
        result_group = self.result.copy()
        test_name = lambda y: "Test Data matched to {}".format(y)
        merger = lambda g, df: (g if df is None else df.reset_index()\
                         .merge(g.reset_index(), how="outer").set_index("x"))

        result_group["y"] = result_group.min_y.apply(test_name)
        result_group = result_group.set_index("y")
        grouped = df.groupby("y_func")
        result_group_index = []
        for y, group in grouped:
            group["y_ideal"] = self.ideal[y]
            group = group.rename(columns={"y_func": "min_y"})
            index_name = test_name(y)
            result_group_index.append(index_name)


            test_group = group.rename({"y_test": result_group[result_group.min_y == y].index.to_list().pop()}, \
                                      axis=1).drop("y_ideal", axis=1)
            #ideal_group = group.rename({"y_ideal": y}, axis=1).drop("y_test", axis=1)

            df_test_group = merger(test_group, df_test_group)
            #df_ideal_group = merger(ideal_group, df_ideal_group)

        df_test_group = df_test_group.sort_index()
        #df_ideal_group = df_ideal_group.sort_index()

        #print("Result Group")
        #print(result_group)
        #print("Test Group\n")
        #print(df_test_group)
        #print("Ideal Group\n")
        #print(df_ideal_group)
        #print("\nNext")
        
        self.result = result_group
        self.real = df_test_group

class TrainingVisualizer(Visualizer):
    def additional_elements(self, index, min_y):
        super().error_bars(self.real[index], self.ideal[min_y])
    
class PlotFactory():
    @staticmethod
    def create_visualizer(result_data, real_data, ideal_data, label=None, vtype=None):
        dict_ = {"train": TrainingVisualizer(result_data, real_data, ideal_data, label), \
                "test": TestVisualizer(result_data, real_data, ideal_data,label)}
        return dict_.get(vtype, dict_.get("train"))
    
    @staticmethod
    def show(label, result_data, real_data, ideal_data, vtype=None):
        v = PlotFactory.create_visualizer(result_data, real_data, ideal_data, label, vtype)

        v.prepare_data()

        data = np.array([])

        for index, row in v.result.iterrows():
            v.create_plot(Data(index, v.real, color="red", vtype="circle"), \
                          Data(row.min_y, v.ideal, line_width=3))
            v.additional_elements(index, row.min_y)
            data = np.append(data, v.plot)
            v.new_plot()

        plot = gridplot(list(np.reshape(data,(2,2))))
        
        show(plot)


In [71]:
# All data must be visualized logically
# Finally, the training data, the test data, the chosen ideal functions as well as the corresponding /
# assigned datasets are visualized under an appropriately chosen representation of the deviation.

# Where possible, create/compile suitable unit-test

# Its design is sensibly object-oriented
# It includes at least one inheritance
# It includes standard- und user-defined exception handlings
# For logical reasons, it makes use of Pandas’ packages as well as data visualization via Bokeh, sqlalchemy,
# as well as others
# Write unit-tests for all useful elements
# Your code needs to be documented in its entirety and also include Documentation Strings, known as ”docstrings“


def main():
    '''
    Initialises the logger.
    '''
    logging.basicConfig(filename="app.log", filemode="a", \
                        format="[%(asctime)s] %(name)-6s - %(levelname)-8s - %(message)s", datefmt='%Y-%m-%d %H:%M:%S')

    global logger
    logger = logging.getLogger()
    logger.setLevel(log_level)

    logger.info("App started")

    # Test the logger in DEBUG and NOTSET mode.
    if logger.level < logging.INFO:
        logger.debug("Debugging")
        logger.warning("Test logger")

    df_train = FileReader.read_csv_to_df("data/train.csv")
    df_ideal = FileReader.read_csv_to_df("data/ideal.csv")

    regression = Regression(df_ideal)
    df_train_mse = df_train.apply(regression.mse)

    df_result = regression.min_value(df_train_mse, df_train)

    
    pu = PersistenceUtil()
    
    pu.save_df_to_db(df_train, "train", r"Y\1 (training func)")
    pu.save_df_to_db(df_ideal, "ideal", r"Y\1 (ideal func)")
    
    chosen_ideal = df_ideal[df_result.min_y.tolist()]
    #print(chosen_ideal)

    data = regression.match("data/test.csv", df_result, chosen_ideal)

    test_table = pu.create_table("test", {"X (test func)": db.Float, "Y (test func)": db.Float, \
                         "Delta Y (test func)": db.Float, "No. of ideal func": db.String(5)}, drop_if_exists=True)
    
    pu.persist(test_table, data)
    
    
    PlotFactory.show("Training Data", df_result, df_train, chosen_ideal)
    PlotFactory.show("Test Data", df_result, data, chosen_ideal, vtype="test")

    print("Done")

if __name__ == '__main__':
    main()

Done


In [72]:
class UnitTestFileReader(unittest.TestCase):
    def test_read_csv_to_df(self):
        try:
            output = FileReader.read_csv_to_df()
            self.assertIsNone(output)

            output = FileReader.read_csv_to_df(file="data/test.csv")
            self.assertIsNotNone(output)
            self.assertEqual(output.size, 100)
            self.assertIsInstance(output, pd.DataFrame)
            self.assertIsInstance(output["y"], pd.Series)
            self.assertEqual(output.index.name, "x")

            output = FileReader.read_csv_to_df(file="doesNotExist.csv")
            self.assertIsNone(output)
        except:
            self.fail("Exception raised")

    def test_csv_line_gen(self):
        try:
            no_gen = FileReader.csv_line_gen()
            self.assertIsNotNone(no_gen)
            try:
                next(no_gen)
                self.fail("Got entry from empty generator")
            except StopIteration:
                pass

            generator = FileReader.csv_line_gen(file="data/train.csv")
            counter = 0
            for line in generator:
                if counter == 4:
                    self.assertEqual(len(line), 5)
                counter += 1
            self.assertEqual(counter, 401)
        except:
            self.fail("Exception raised")

            
            
            
            
            
            
            
class UnitTestRegression(unittest.TestCase):
    def test_constructor(self):
        try:
            Regression()
            self.fail("No Exception raised")
        except AssertionError as e:
            raise e
        except RegressionException:
            pass
        except Exception as e:
            self.fail("Wrong Exception raised: {}".format(type(e).__name__))
            
        x_list = [x for x in range(1,101,1)]
        y_list = ["y{}".format(x) for x in range(1,101,1)]
        
        df_ideal = pd.DataFrame(20*np.random.random_sample(size=(100,100))-10, columns=[x for x in range(100)])
        df_ideal["y"] = y_list
        df_ideal = df_ideal.set_index("y")

        try:
            Regression(df_ideal)
            self.fail("No RegressionException raised")
        except RegressionException as e:
            #self.assertEqual("NaN values in index column.", e.__str__())
            self.assertEqual("No number values in index column.", e.__str__())

        multi_index = [(float(x/10),20*np.random.random_sample()-10) for x in range(100)]
        df_ideal["x"] = multi_index
        df_ideal = df_ideal.set_index("x")
        
        try:
            Regression(df_ideal)
            self.fail("No RegressionException raised")
        except RegressionException as e:
            #self.assertEqual("NaN values in index column.", e.__str__())
            self.assertEqual("Multi index provided", e.__str__())
        

        df_ideal["x"] = x_list
        df_ideal = df_ideal.set_index("x")
        
        regression = Regression(df_ideal)
        
        self.assertIsInstance(regression, Regression)

    def test_mse(self):
        df = pd.DataFrame(20*np.random.random_sample(size=(100, 3))-10, columns=["x", "y", "y2"]).set_index("x")
        df2 = df.copy().drop("y2", axis=1)
        regression = Regression(df2)
        try:
            regression.mse(df)
            self.fail("No RegressionException raised")
        except RegressionException:
            pass
        except Exception as e:
            self.fail("Wrong Exception raised: {}".format(type(e).__name__))

        self.assertEqual(regression.mse(df2).y, 0)

    def test_min_value(self):
        x_list = [(float(x/10),20*np.random.random_sample()-10) for x in range(-50, 50, 1)]
        y_list = ["y{}".format(x) for x in range(1,101,1)]

        df_ideal = pd.DataFrame(20*np.random.random_sample(size=(100,100))-10, columns=y_list)
        df_ideal["x"] = [float(x/10) for x in range(-50, 50, 1)]
        df_ideal = df_ideal.set_index("x")
        
        df = pd.DataFrame(np.random.random_sample(size=(100,100)), columns=y_list)
        df["y"] = y_list
        df = df.set_index("y")

        diff = pd.DataFrame(x_list, columns=["x", "z"]).set_index("x")

        regression = Regression(df_ideal)
        
        try:
            regression.min_value(df, diff)
            self.fail("No RegressionException raised")
        except RegressionException as e:
            self.assertEqual("Found NaN values in max_deviation column.", e.__str__())
        except Exception as e:
            self.fail("Wrong Exception raised: {}".format(type(e).__name__))
        
        df.index = y_list
        diff = pd.DataFrame(np.random.random_sample(size=(100,100)), columns=y_list)
        diff["x"] = [float(x/10) for x in range(-50, 50, 1)]
        diff = diff.set_index("x")
        
        min_value = regression.min_value(df, diff)
        
        self.assertFalse(min_value.isnull().values.any())

    def test_match(self):
        y_list = ["y{}".format(x) for x in range(1,11,1)]
        index_list = ["y{}".format(np.random.randint(1,10)) for i in range(0,400)]

        df_ideal = pd.DataFrame(20*np.random.random_sample(size=(400,10))-10, columns=y_list)
        df_ideal["x"] = [float(x/10) for x in range(-200, 200, 1)]
        df_ideal = df_ideal.set_index("x")

        df = pd.DataFrame(np.random.random_sample(size=(400,10)), columns=y_list)
        df.index = index_list
        diff = pd.DataFrame(np.random.random_sample(size=(400,10)), columns=y_list)
        diff["x"] = [float(x/10) for x in range(-200, 200, 1)]
        diff = diff.set_index("x")

        regression = Regression(df_ideal)
        
        min_value = regression.min_value(df, diff)
        chosen_ideal = df_ideal[min_value.min_y.tolist()]
        
        match = regression.match("data/test.csv", min_value, chosen_ideal)
        
        self.assertIsNotNone(match)
        self.assertNotEqual(len(match),0)

class UnitTestPersistenceUtil(unittest.TestCase):
    def test_df_column_name(self):
        pu = PersistenceUtil(db.create_engine(f"sqlite:///data/unittest.db"))
        
        tests = {"test": ["abc", "test"], \
                 "t3st": ["abc", "t3st"], \
                 "ty3st": ["abc", "tabcst"], \
                 "ty3st": [r"a\1c", "ta3cst"], \
                 "y1y4y67": ["abc", "abcabcabc7"], \
                 "y1 y4 y67": [r"a\1b\1c\1d\1", "a1b1c1d1 a4b4c4d4 a6b6c6d67"], \
                 "tY1st": ["abc", "tY1st"]}
        
        for k in tests:
            self.assertEqual(pu.df_column_name(k,tests[k][0]), tests[k][1])

    def test_save_df_to_db(self): # df, table, regex_column_name):
        y_list = ["y{}".format(x) for x in range(1,11,1)]
        ut_list = ["u_t{}".format(x) for x in range(1,11,1)]
        pu = PersistenceUtil(db.create_engine(f"sqlite:///data/unittest.db"))
        df = pd.DataFrame(np.random.random_sample(size=(400,10)), columns=y_list)
        df.index.name = "x"
        table_name = "u_test_01"
        
        rows_count = pu.save_df_to_db(df, table_name, r"u_t\1")
        df_db = pu.read_df(table_name).set_index("X")

        self.assertEqual(df_db.columns.to_list(), ut_list)

        df_db.columns = y_list
        df_db.index.name = df_db.index.name.lower()
        
        self.assertEqual(len(df.index), rows_count)
        self.assertEqual(len(df.index), len(df_db.index))
        self.assertEqual(len(df.columns), len(df_db.columns))

        self.assertTrue(df.equals(df_db))
    
    def test_create_and_persist(self): # table, columns):
        data_list = [\
                     {"col_str": "Test value", "col_int": -3}, \
                     {"col_str": "Test value 2", "col_int": 5.0} \
                    ]
        
        table_name = "test_create"
        pu = PersistenceUtil(db.create_engine(f"sqlite:///data/unittest.db"))
        try:
            table = pu.create_table(table_name, {"col_str": db.String(10), "col_int": db.Integer}, \
                                    drop_if_exists=True)
            self.assertEqual(table, table_name)
            pu.persist(table, data_list)

            df = pu.read_df(table_name).set_index("col_str")
            df_dict = pd.DataFrame(data_list).set_index("col_str")
            df_dict["col_int"] = df_dict.col_int.apply(int)
            self.assertTrue(df.equals(df_dict))

        except Exception as e:
            self.fail("Exception raised: {}".format(type(e).__name__))
        
        data_list = [\
                     {"col_str": "Test value 3", "col_int": -3.5}, \
                     {"col_str": "Test value 4", "col_int": "String"} \
                    ]
        #print(data_list)
        try:
            pu.persist(table, data_list)
            df = pu.read_df(table_name).set_index("col_str")
            print(df)
        except Exception as e:
            print(e)

class UnitTestVisualizer(unittest.TestCase):
    def test_to_url(self):
        tests = {"I lov' #coo$k-ies.": "i_lov_cookies", \
               " ": "_", \
               "": ""}
        
        for k in tests:
            self.assertEqual(Visualizer.to_url(k), tests[k])
    
class UnitTestPlotfactory(unittest.TestCase):
    def test_create_visualizer(self):
        df = pd.DataFrame(np.random.randint(0,20,size=(5, 3)), columns=["x", "y", "z"]).set_index("x")

        test = TestVisualizer(df, df, df)
        train = TrainingVisualizer(df, df, df)
        tests = {"I lov' #coo$k-ies.": train, \
                 "test": test, \
               None: train, \
               "train": train}
        
        for k in tests:
            self.assertIsInstance(PlotFactory.create_visualizer(df, df, df, vtype=k), type(tests[k]))






if __name__ == '__main__':
    unittest.main(argv=[''], verbosity=2, exit=False)


test_csv_line_gen (__main__.UnitTestFileReader) ... ok
test_read_csv_to_df (__main__.UnitTestFileReader) ... ok
test_create_and_persist (__main__.UnitTestPersistenceUtil) ... ok
test_df_column_name (__main__.UnitTestPersistenceUtil) ... ok
test_save_df_to_db (__main__.UnitTestPersistenceUtil) ... ok
test_create_visualizer (__main__.UnitTestPlotfactory) ... ok
test_constructor (__main__.UnitTestRegression) ... ok
test_match (__main__.UnitTestRegression) ... 

Exception found for data [{'col_str': 'Test value 3', 'col_int': -3.5}, {'col_str': 'Test value 4', 'col_int': 'String'}]
invalid literal for int() with base 10: 'String'


ok
test_min_value (__main__.UnitTestRegression) ... ok
test_mse (__main__.UnitTestRegression) ... ok
test_to_url (__main__.UnitTestVisualizer) ... ok

----------------------------------------------------------------------
Ran 11 tests in 0.387s

OK
